# Module 04 - Deploy the managed online endpoint

Serve the model and run a what-if scenario. See [README.md](README.md).

In [ ]:
# Tutorial setup: find the repo root and make the repo code importable.
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    return start


REPO = find_repo_root(Path.cwd())
for sub in ["src", "data", "webapp/backend"]:
    sys.path.insert(0, str(REPO / sub))
print("Repo root:", REPO)

## Deploy the endpoint (optional, ~6-10 min)

If `etrm-forecast` already exists you can skip this. Uncomment to deploy.

In [ ]:
import subprocess

# proc = subprocess.run(
#     [sys.executable, str(REPO / "src" / "deploy" / "deploy_endpoint.py"),
#      "--model-dir", str(REPO / "model_artifacts")],
#     cwd=str(REPO), capture_output=True, text=True,
# )
# print(proc.stdout[-2000:])
print("Skip if the endpoint already exists.")

## Call the endpoint for a 24-hour forecast

In [ ]:
import json, requests

res = json.loads((REPO / ".azure-resources.json").read_text())
url, key = res["aml_endpoint_url"], res["aml_endpoint_key"]
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {key}"}

base = requests.post(url, headers=headers, json={"horizon_hours": 24}, timeout=30).json()
print("Units:", base["units"], "| Model:", base["model"])
print("Summary:", json.dumps(base["summary"], indent=2))

## Run a cold-snap scenario

Temperature down 8C and demand up 10 percent. Same model, higher prices.

In [ ]:
scenario = requests.post(
    url, headers=headers,
    json={"horizon_hours": 24, "temperature_offset_c": -8, "demand_multiplier": 1.10},
    timeout=30,
).json()
print("Baseline avg:", base["summary"]["avg"])
print("Cold-snap avg:", scenario["summary"]["avg"])

## Next

Continue to [Module 05](../05-foundry-agent/README.md).